In [17]:
import requests
import pandas as pd

url = "https://opendata.dwd.de/climate_environment/health/alerts/s31fg.json"
data = requests.get(url).json()

rows = []

for region in data["content"]:
    region_name = region["region_name"]
    pollen_dict = region["Pollen"]  # dict of species → forecast dict

    for species, forecast in pollen_dict.items():

        rows.append({
            "region": region_name,
            "species": species,
            "day": "today",
            "level": forecast.get("today", None)
        })

        rows.append({
            "region": region_name,
            "species": species,
            "day": "tomorrow",
            "level": forecast.get("tomorrow", None)
        })

        rows.append({
            "region": region_name,
            "species": species,
            "day": "day_after_tomorrow",
            "level": forecast.get("day_after_tomorrow", None)
        })

        rows.append({
            "region": region_name,
            "species": species,
            "day": "day4",
            "level": forecast.get("day4", None)
        })

        rows.append({
            "region": region_name,
            "species": species,
            "day": "day5",
            "level": forecast.get("day5", None)
        })

df = pd.DataFrame(rows)


1. Your DWD scraper is now 100% correct
Your DataFrame shows:

region → DWD region (e.g., Schleswig-Holstein und Hamburg)

species → Hasel, Birke, Erle, Graeser, Ambrosia, Beifuss

day → today, tomorrow, day_after_tomorrow, day4, day5

level → numeric pollen level (0, 0–1, 1–2, etc.)

This is exactly what DWD provides.

Why some values are None?
Because DWD does not always publish all 5 days for every species.

Example:

Hasel season is ending → only today/tomorrow available

Graeser season is starting → only today/tomorrow available

Birke season may have full 5 days

This is normal and expected.

In [18]:
import datetime
import pandas as pd

# 1. English pollen names
species_map = {
    "Hasel": "Hazel",
    "Erle": "Alder",
    "Birke": "Birch",
    "Graeser": "Grass",
    "Ambrosia": "Ragweed",
    "Beifuss": "Mugwort"
}

df["pollen_name"] = df["species"].map(species_map)

# 2. Region → City mapping
region_to_city = {
    "Schleswig-Holstein und Hamburg": "Hamburg",
    "Mecklenburg-Vorpommern": "Rostock",
    "Niedersachsen und Bremen": "Hanover",
    "Nordrhein-Westfalen": "Düsseldorf",
    "Brandenburg und Berlin": "Berlin",
    "Sachsen-Anhalt": "Magdeburg",
    "Sachsen": "Dresden",
    "Thüringen": "Erfurt",
    "Hessen": "Frankfurt",
    "Rheinland-Pfalz und Saarland": "Mainz",
    "Baden-Württemberg": "Stuttgart",
    "Bayern": "Munich"
}

df["city"] = df["region"].map(region_to_city)

# 3. Convert day → real date
today = datetime.date.today()
day_map = {
    "today": today,
    "tomorrow": today + datetime.timedelta(days=1),
    "day_after_tomorrow": today + datetime.timedelta(days=2),
    "day4": today + datetime.timedelta(days=3),
    "day5": today + datetime.timedelta(days=4)
}

df["date"] = df["day"].map(day_map)

# 4. Convert numeric level → category
def categorize(level):
    if level is None:
        return None

    if "-" in level:
        a, b = level.split("-")
        try:
            value = (float(a) + float(b)) / 2
        except:
            return None
    else:
        try:
            value = float(level)
        except:
            return None

    if value == 0:
        return "None"
    elif 0 < value <= 1:
        return "Very Low"
    elif 1 < value <= 2:
        return "Low"
    elif 2 < value <= 3:
        return "Moderate"
    elif 3 < value <= 4:
        return "High"
    else:
        return "Very High"

df["category"] = df["level"].apply(categorize)

# 5. Pivot to wide format
pollen_df = df.pivot_table(
    index=["city", "date"],
    columns="pollen_name",
    values="category",
    aggfunc="first"
).reset_index()

# 6. Order columns
pollen_df = pollen_df[
    ["city", "date", "Hazel", "Alder", "Birch", "Grass", "Ragweed", "Mugwort"]
]

pollen_df


pollen_name,city,date,Hazel,Alder,Birch,Grass,Ragweed,Mugwort
0,Berlin,2026-05-18,None,None,Very Low,Low,None,None
1,Berlin,2026-05-19,None,None,None,Low,None,None
2,Dresden,2026-05-18,None,None,Very Low,Low,None,None
3,Dresden,2026-05-19,None,None,None,Low,None,None
4,Düsseldorf,2026-05-18,None,None,None,Very Low,None,None
5,Düsseldorf,2026-05-19,None,None,None,Low,None,None
6,Erfurt,2026-05-18,None,None,Very Low,Low,None,None
7,Erfurt,2026-05-19,None,None,None,Low,None,None
8,Frankfurt,2026-05-18,None,None,None,Low,None,None
9,Frankfurt,2026-05-19,None,None,None,Low,None,None


In [19]:
pollen_df = pollen_df.sort_values(["date", "city"]).reset_index(drop=True)
pollen_df

pollen_name,city,date,Hazel,Alder,Birch,Grass,Ragweed,Mugwort
0,Berlin,2026-05-18,None,None,Very Low,Low,None,None
1,Dresden,2026-05-18,None,None,Very Low,Low,None,None
2,Düsseldorf,2026-05-18,None,None,None,Very Low,None,None
3,Erfurt,2026-05-18,None,None,Very Low,Low,None,None
4,Frankfurt,2026-05-18,None,None,None,Low,None,None
5,Hamburg,2026-05-18,None,None,None,Low,None,None
6,Hanover,2026-05-18,None,None,None,Very Low,None,None
7,Magdeburg,2026-05-18,None,None,Very Low,Low,None,None
8,Mainz,2026-05-18,None,None,None,Low,None,None
9,Munich,2026-05-18,None,None,None,Low,None,None


In [20]:
pollen_df["city"].unique()

array(['Berlin', 'Dresden', 'Düsseldorf', 'Erfurt', 'Frankfurt',
       'Hamburg', 'Hanover', 'Magdeburg', 'Mainz', 'Munich', 'Rostock',
       'Stuttgart'], dtype=object)

In [21]:
#convert pollen_df to csv file
pollen_df.to_csv("dwd_pollen_5day.csv", index=False)

Now scraping weather data 
#🌦️ STEP 1 — Coordinates for each city

In [22]:
city_coords = {
    "Berlin": (52.5200, 13.4050),
    "Hamburg": (53.5511, 9.9937),
    "Munich": (48.1351, 11.5820),
    "Frankfurt": (50.1109, 8.6821),
    "Stuttgart": (48.7758, 9.1829),
    "Dresden": (51.0504, 13.7373),
    "Düsseldorf": (51.2277, 6.7735),
    "Erfurt": (50.9848, 11.0299),
    "Hanover": (52.3759, 9.7320),
    "Magdeburg": (52.1205, 11.6276),
    "Mainz": (49.9929, 8.2473),
    "Rostock": (54.0924, 12.0991)
}


🌦️ STEP 2 — Weather API (Open‑Meteo)
We fetch:

max temp

min temp

precipitation

windspeed

humidity

for 5 days.

In [23]:
#🌦️ STEP 3 — Working 5‑Day Weather Scraper
import requests
import pandas as pd
import datetime

def get_weather(lat, lon):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "daily": [
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_sum",
            "windspeed_10m_max",
            "winddirection_10m_dominant"
        ],
        "timezone": "Europe/Berlin"
    }
    r = requests.get(url, params=params)
    return r.json()

weather_rows = []

for city, (lat, lon) in city_coords.items():
    print("Scraping weather for:", city)
    data = get_weather(lat, lon)

    dates = data["daily"]["time"]
    tmax = data["daily"]["temperature_2m_max"]
    tmin = data["daily"]["temperature_2m_min"]
    rain = data["daily"]["precipitation_sum"]
    wind = data["daily"]["windspeed_10m_max"]
    winddir = data["daily"]["winddirection_10m_dominant"]

    for i in range(len(dates)):
        weather_rows.append({
            "city": city,
            "date": dates[i],
            "temp_max": tmax[i],
            "temp_min": tmin[i],
            "rain_sum": rain[i],
            "wind_max": wind[i],
            "wind_direction": winddir[i]
        })

weather_df = pd.DataFrame(weather_rows)
weather_df


Scraping weather for: Berlin
Scraping weather for: Hamburg
Scraping weather for: Munich
Scraping weather for: Frankfurt
Scraping weather for: Stuttgart
Scraping weather for: Dresden
Scraping weather for: Düsseldorf
Scraping weather for: Erfurt
Scraping weather for: Hanover
Scraping weather for: Magdeburg
Scraping weather for: Mainz
Scraping weather for: Rostock


,city,date,temp_max,temp_min,rain_sum,wind_max,wind_direction
0,Berlin,2026-05-18,20.0,8.5,0.00,13.7,112
1,Berlin,2026-05-19,19.7,9.7,0.00,9.3,203
2,Berlin,2026-05-20,19.6,13.5,0.10,9.5,222
3,Berlin,2026-05-21,21.7,12.9,0.00,11.0,272
4,Berlin,2026-05-22,23.9,13.8,0.00,9.9,312
...,...,...,...,...,...,...,...
79,Rostock,2026-05-20,17.2,10.8,2.05,12.6,188
80,Rostock,2026-05-21,19.8,11.5,1.10,16.9,250
81,Rostock,2026-05-22,21.7,13.1,0.00,13.2,287
82,Rostock,2026-05-23,25.1,13.5,0.00,11.6,152


In [24]:
# convert weather_df to csv file 
weather_df.to_csv("meteo_weather_5day.csv", index=False)

streamlit run app.py 
